# Bronze -> Silver (Dia 2)

Limpeza, tipagem e joins basicos da camada Bronze para a Silver.

**Executar dentro do Fabric**, anexado ao Lakehouse `lh_olist`.

Entrada: tabelas `brz_*` (CSV bruto ingerido no Dia 1).
Saida: tabelas `slv_*` em Delta (limpas e tipadas).

In [ ]:
from pyspark.sql import functions as F

LAKEHOUSE = "lh_olist"  # ajustar se necessario
# Lakehouse com esquema (schema-enabled): as tabelas ficam sob dbo.
# Se for um Lakehouse legado (sem esquema), deixe SCHEMA = "".
SCHEMA = "dbo"

def tbl(name: str) -> str:
    """Qualifica o nome da tabela com o esquema, quando houver."""
    return f"{SCHEMA}.{name}" if SCHEMA else name

TABLES = ["orders", "order_items", "order_payments", "customers", "sellers", "products"]

In [ ]:
# 1. Ler Bronze
df = {t: spark.read.table(tbl(f"brz_{t}")) for t in TABLES}
for t, d in df.items():
    print(t, d.count(), "linhas")

In [ ]:
# 2. Tipagem e limpeza (exemplo: orders)
ts_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
slv_orders = df["orders"]
for c in ts_cols:
    slv_orders = slv_orders.withColumn(c, F.to_timestamp(c))
slv_orders = slv_orders.dropDuplicates(["order_id"])

In [ ]:
# 3. Tipar valores monetarios em order_items / order_payments
slv_order_items = (
    df["order_items"]
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("freight_value", F.col("freight_value").cast("double"))
)
slv_order_payments = (
    df["order_payments"]
    .withColumn("payment_value", F.col("payment_value").cast("double"))
)

In [ ]:
# 4. Gravar Silver em Delta
slv = {
    "orders": slv_orders,
    "order_items": slv_order_items,
    "order_payments": slv_order_payments,
    "customers": df["customers"].dropDuplicates(["customer_id"]),
    "sellers": df["sellers"].dropDuplicates(["seller_id"]),
    "products": df["products"].dropDuplicates(["product_id"]),
}
for name, d in slv.items():
    d.write.mode("overwrite").format("delta").saveAsTable(tbl(f"slv_{name}"))
    print("gravado slv_" + name)